In [1]:
from copy import deepcopy

import fasttext
import fasttext.util
import matplotlib.pyplot as plt
from matplotlib.image import imread
from mpl_toolkits import mplot3d
from matplotlib import gridspec
from PIL import Image
import io
import os
from urllib.request import urlopen
from skimage.segmentation import mark_boundaries
from nltk.tokenize import RegexpTokenizer
from torchinfo import summary
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import requests
from scipy.stats import norm
import torch

from sklearn.metrics import classification_report
from torch.utils.tensorboard import SummaryWriter

from torchvision import datasets, transforms

2026-03-26 09:42:57.387751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774518177.754178      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774518177.851830      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774518178.916163      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774518178.916217      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774518178.916221      55 computation_placer.cc:177] computation placer alr

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## Код для обучения

In [4]:
class callback():
    def __init__(self, writer, dataset, loss_function, id2tag = None, ind_to_word = None, delimeter = 100, batch_size=64, num_exampels=3):
        self.step = 0
        self.writer = writer
        self.delimeter = delimeter
        self.loss_function = loss_function
        self.batch_size = batch_size
        self.num_exampels = num_exampels

        self.dataset = dataset
        self.id2tag = id2tag
        self.ind_to_word = ind_to_word

    def forward(self, model, loss):
        self.step += 1
        self.writer.add_scalar('LOSS/train', loss, self.step)
        
        if self.step % self.delimeter == 0:
            
            batch_generator = torch.utils.data.DataLoader(dataset = self.dataset, 
                                                          batch_size=self.batch_size, shuffle=True)
            
            pred = []
            real = []
            test_loss = 0
            model.eval()

            for it, (x_batch, y_batch) in enumerate(batch_generator):
                device = next(model.parameters()).device
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)

                logits, targets = model(x_batch)
                logits = logits.permute(0, 2, 1)   # (batch, vocab_size, seq_len-1)
                test_loss += self.loss_function(logits, targets).cpu().item() * len(x_batch)
            
            test_loss /= len(self.dataset)
            
            self.writer.add_scalar('LOSS/test', test_loss, self.step)

            # x_batch, y_batch = next(iter(batch_generator))
            # x_batch = x_batch.to(model.device)
            # with torch.no_grad():
            #     outputs = model(x_batch)
            #     preds = torch.argmax(outputs, dim=1)

            # text = f"Examples at step {self.step}\n"
            # for i in range(self.num_exampels):
            #     tokens = [self.ind_to_word.get(idx.item(), '[UNK]') for idx in x_batch[i]]
            #     true_tags = [self.id2tag.get(idx.item(), '?') for idx in y_batch[i] if idx != -100]

            #     pred_tags = []
            #     for j, idx in enumerate(y_batch[i]):
            #         if idx != -100:
            #             pred_tags.append(self.id2tag.get(preds[i][j].item(), '?'))
            #     text += f"\nExample {i+1}:\n"
            #     text += f"Tokens: {' '.join(tokens)}\n"
            #     text += f"True:   {' '.join(true_tags)}\n"
            #     text += f"Pred:   {' '.join(pred_tags)}\n"

            # self.writer.add_text('Examples', text, self.step)


          
    def __call__(self, model, loss):
        return self.forward(model, loss)

In [5]:
def train_on_batch(model, x_batch, y_batch, optimizer, loss_function):
    model.train()
    optimizer.zero_grad()
    device = next(model.parameters()).device
    
    # Модель возвращает (logits, targets)
    logits, targets = model(x_batch.to(device))
    # Приводим логиты к формату (batch, vocab_size, seq_len) для CrossEntropyLoss
    logits = logits.permute(0, 2, 1)
    loss = loss_function(logits, targets)
    loss.backward()
    optimizer.step()
    return loss.cpu().item()

In [6]:
def train_epoch(train_generator, model, loss_function, optimizer, callback = None):
    epoch_loss = 0
    total = 0
    for it, (batch_of_x, batch_of_y) in enumerate(train_generator):
        batch_loss = train_on_batch(model, batch_of_x, batch_of_y, optimizer, loss_function)
        
        if callback is not None:
            with torch.no_grad():
                callback(model, batch_loss)
            
        epoch_loss += batch_loss*len(batch_of_x)
        total += len(batch_of_x)
    
    return epoch_loss/total

In [7]:
def trainer(count_of_epoch, 
            batch_size, 
            dataset,
            model, 
            loss_function,
            optimizer,
            lr = 0.001,
            callback = None):

    optima = optimizer(model.parameters(), lr=lr)
    
    iterations = tqdm(range(count_of_epoch), desc='epoch')
    iterations.set_postfix({'train epoch loss': np.nan})
    for it in iterations:
        batch_generator = tqdm(
            torch.utils.data.DataLoader(dataset=dataset, 
                                        batch_size=batch_size, 
                                        shuffle=True, pin_memory=True), 
            leave=False, total=len(dataset)//batch_size+(len(dataset)%batch_size>0))
        
        epoch_loss = train_epoch(train_generator=batch_generator, 
                    model=model, 
                    loss_function=loss_function, 
                    optimizer=optima, 
                    callback=callback)
        
        iterations.set_postfix({'train epoch loss': epoch_loss})

In [8]:
def testing_on_test_sample(model, dataset_test_pt):
    batch_generator = torch.utils.data.DataLoader(
        dataset=dataset_test_pt,
        batch_size=64,
        pin_memory=True,
        shuffle=False
    )
    correct = 0
    total = 0
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        for x_batch, _ in batch_generator:   # y_batch не нужен
            x_batch = x_batch.to(device)
            logits, targets = model(x_batch)   # (batch, seq_len-1, vocab_size), (batch, seq_len-1)
            preds = torch.argmax(logits, dim=2)
            mask = (targets != 0)           # игнорируем специальные токены
            correct += (preds[mask] == targets[mask]).sum().item()
            total += mask.sum().item()
    print(f"Token-level accuracy: {correct / total:.4f} ({correct}/{total})")

## Загрузка датасета

In [9]:
import pandas as pd

dataset_len = 500000
num = 0

data = []
with open('/kaggle/input/datasets/bobrzol123/twiter/twitter.csv', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i == 0:
            continue
        parts = line.strip().split(';')
        if not parts:
            continue
        first = parts[0].strip('"')
        if ',' in first:
            tag, message = first.split(',', 1)
            try:
                tag = int(tag)
            except ValueError:
                continue
            data.append((tag, message))
        
        num += 1
        if num > dataset_len:
            break

df = pd.DataFrame(data, columns=['tag', 'message'])
print(df.head())

   tag                                            message
0    0           is so sad for my APL friend.............
1    0                   I missed the New Moon trailer...
2    1                            omg its already 7:30 :O
3    0  .. Omgaga. Im sooo  im gunna CRy. I've been at...
4    0       i think mi bf is cheating on me!!!       T_T


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
max_length_sentence = 20

In [12]:
dataset = []
tokens = []
word_to_ind = {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[SOS]': 4}

for sent in df['message']:
    encoded = tokenizer(sent, max_length=max_length_sentence, truncation=True, padding='max_length')
    arr = []
    for token in encoded.tokens():
        if token not in word_to_ind:
            word_to_ind[token] = len(word_to_ind)
            tokens.append(word_to_ind[token])

    for token in encoded.tokens():
        arr.append(word_to_ind[token])
    dataset.append(arr)

del tokens

In [13]:
import psutil
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 31.35 GB
Доступно: 28.75 GB
Используется: 2.15 GB
Процент использования: 8.3%


#### Итого у нас 17 частей речи есть датасет, вида [предложение, [части речи слов]]

In [14]:
from sklearn.model_selection import train_test_split

dataset_train, dataset_test = train_test_split(dataset, test_size=0.2, random_state=42)

## Создаём Rnn модель

In [15]:
dataset_train = torch.tensor(dataset_train)
dataset_test = torch.tensor(dataset_test)

In [16]:
dataset_train_pt = torch.utils.data.TensorDataset(dataset_train, dataset_train)
dataset_test_pt  = torch.utils.data.TensorDataset(dataset_test,  dataset_test)

In [17]:
# Получаем статистику памяти
mem = psutil.virtual_memory()

print(f"Всего: {mem.total / (1024**3):.2f} GB")
print(f"Доступно: {mem.available / (1024**3):.2f} GB")
print(f"Используется: {mem.used / (1024**3):.2f} GB")
print(f"Процент использования: {mem.percent}%")

Всего: 31.35 GB
Доступно: 28.67 GB
Используется: 2.23 GB
Процент использования: 8.5%


In [18]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [19]:
padding_idx_glob=word_to_ind['[PAD]']
print(padding_idx_glob)

0


In [20]:
import torch
import torch.nn as nn


class Encoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 emb_dim=128,
                 hidden_dim=256,
                 num_layers=2,
                 bottleneck_dim=128,
                 bidirectional=True,
                 dropout=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.num_directions = 2 if bidirectional else 1
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        # Линейный слой для получения вектора фиксированного размера
        self.projection = nn.Linear(hidden_dim * self.num_directions, bottleneck_dim)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)  # (batch, seq_len, emb_dim)

        _, (h, _) = self.lstm(emb)
        # h: (num_layers * num_directions, batch, hidden_dim)

        # Берём последний слой (конкатенируем оба направления, если есть)
        start = (self.num_layers - 1) * self.num_directions
        last_layer_h = h[start:start + self.num_directions]  # (num_directions, batch, hidden_dim)
        last_layer_h = last_layer_h.transpose(0, 1).reshape(last_layer_h.size(1), -1)
        # last_layer_h: (batch, hidden_dim * num_directions)

        bottleneck = self.projection(last_layer_h)  # (batch, bottleneck_dim)
        return bottleneck


class Decoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 emb_dim=128,
                 hidden_dim=256,
                 latent_dim=128,
                 num_layers=2,
                 dropout=0.0):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.logits = nn.Linear(hidden_dim, vocab_size)

        # Линейные слои для преобразования latent в начальные состояния всех слоёв
        self.latent_to_h0 = nn.Linear(latent_dim, hidden_dim * num_layers)
        self.latent_to_c0 = nn.Linear(latent_dim, hidden_dim * num_layers)

    def forward(self, word_ix, latent_vector=None, h=None, c=None):
        """
        Args:
            word_ix: (batch, seq_len) – входные токены
            latent_vector: (batch, latent_dim) – используется, если h и c не заданы
            h, c: начальные состояния (num_layers, batch, hidden_dim)
        Returns:
            logits: (batch, seq_len, vocab_size)
            (h, c): обновлённые состояния
        """
        if h is None or c is None:
            # Инициализируем состояния из latent_vector
            batch_size = latent_vector.size(0)
            h0 = self.latent_to_h0(latent_vector).view(
                self.num_layers, batch_size, self.hidden_dim
            )
            c0 = self.latent_to_c0(latent_vector).view(
                self.num_layers, batch_size, self.hidden_dim
            )
        else:
            h0, c0 = h, c

        emb = self.embedding(word_ix)
        lstm_out, (h_n, c_n) = self.lstm(emb, (h0, c0))
        logits = self.logits(lstm_out)

        return logits, (h_n, c_n)


class Autoencoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 emb_dim=128,
                 hidden_dim=256,
                 bottleneck_dim=128,
                 num_layers=2,
                 bidirectional=True,
                 dropout=0.0):
        super().__init__()
        self.encoder = Encoder(
            vocab_size, emb_dim, hidden_dim, num_layers,
            bottleneck_dim, bidirectional, dropout
        )
        self.decoder = Decoder(
            vocab_size, emb_dim, hidden_dim, bottleneck_dim,
            num_layers, dropout
        )

    def forward(self, x):
        """
        Обучение с teacher forcing.
        Args:
            x: (batch, seq_len) – входная последовательность
        Returns:
            logits: (batch, seq_len-1, vocab_size)
            targets: (batch, seq_len-1) – целевые токены (сдвинутые на 1)
        """
        bottleneck = self.encoder(x)
        dec_input = x[:, :-1]
        logits, _ = self.decoder(dec_input, latent_vector=bottleneck)
        return logits, x[:, 1:]

In [21]:
from torch.nn import DataParallel

model = Autoencoder(
    vocab_size=len(word_to_ind),
    emb_dim=300,
    hidden_dim=256,
    bottleneck_dim=256,
    num_layers=2,
    dropout=0.6
)

In [22]:
import fasttext.util
import contextlib
import os
import sys

with open(os.devnull, 'w') as devnull:
    with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
        import fasttext.util
        fasttext.util.download_model('en', if_exists='ignore')
        ft = fasttext.load_model('cc.en.300.bin')

embeddings = torch.zeros(len(word_to_ind), 300)
for word, idx in word_to_ind.items():
    try:
        embeddings[idx] = ft.get_word_vector(word)
    except:
        embeddings[idx] = torch.randn(300)

model.encoder.embedding.weight.data = embeddings
model.decoder.embedding.weight.data = embeddings

model.encoder.embedding.weight.requires_grad = False
model.decoder.embedding.weight.requires_grad = False

In [23]:
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Используем {torch.cuda.device_count()} GPU")

model = model.to(device)

Используем 2 GPU


In [24]:
testing_on_test_sample(model, dataset_test_pt)

Token-level accuracy: 0.0000 (20/1676769)


In [25]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      | 107812 KiB | 396676 KiB | 504321 MiB | 504216 MiB |
|       from large pool | 105152 KiB | 393976 KiB | 496505 MiB | 496402 MiB |
|       from small pool |   2660 KiB |   5012 KiB |   7815 MiB |   7813 MiB |
|---------------------------------------------------------------------------|
| Active memory         | 107812 KiB | 396676 KiB | 504321 MiB | 504216 MiB |
|       from large pool | 105152 KiB | 393976 KiB | 496505 MiB |

In [26]:
loss_function = torch.nn.CrossEntropyLoss(
    ignore_index=0
)
optimizer = torch.optim.Adam

In [27]:
writer = SummaryWriter(log_dir = 'model-lstm-1')

ind_to_word = {v: k for k, v in word_to_ind.items()}
call = callback(writer, dataset_test_pt, loss_function, delimeter = 6200)

trainer(count_of_epoch=1, 
        batch_size=64, 
        dataset=dataset_train_pt,
        model=model, 
        loss_function=loss_function,
        optimizer = optimizer,
        lr=0.001,
        callback=call)

epoch:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/6249 [00:00<?, ?it/s]

In [28]:
testing_on_test_sample(model, dataset_test_pt)

Token-level accuracy: 0.2850 (477818/1676769)
